# Creating Constraints

Constraints are created and at the same time assigned to the model using the function 

```
model.add_constraints
```
where `model` is a `linopy.Model` instance. Again, we want to understand this function and its argument. So, let's create a model first.

In [ ]:
from linopy import Model

m = Model()

Given a variable `x` which has to by lower than 10/3, the constraint would be formulated as 

$$ x \le \frac{10}{3} $$

or

$$ 3 x \le 10 $$
 
or 

$$ x - \frac{3}{10} \le 0 $$


of which all formulations can be written out with linopy just like that. 

In [ ]:
x = m.add_variables(name="x")

When applying one of the operators `<=`, `>=`, `==` to the expression, an unassigned constraint is built:

In [ ]:
con = 3 * x <= 10
con

Unasssigned means, it is not yet added to the model. We can inspect the elements of the anonymous constraint: 

In [ ]:
con.lhs

In [ ]:
con.rhs

We can now add the constraint to the model by passing the unassigned `Constraint` to the `.add_constraint` function. 

In [ ]:
c = m.add_constraints(con, name="my-constraint")
c

The same output would be generated if passing lhs, sign and rhs as separate arguments to the function:

In [ ]:
m.add_constraints(3 * x <= 10, name="the-same-constraint")

Note that the return value of the operation is a `Constraint` which contains the reference labels to the constraints in the optimization model. Also is redirects to its lhs, sign and rhs, for example we can call

In [ ]:
c.lhs

to inspect the lhs of a defined constraint.

When moving the constant value to the left hand side in the initialization, it will be pulled to the right hand side as soon as the constraint is defined

In [ ]:
3 * x - 10

In [ ]:
3 * x - 10 <= 0

Like this, the all defined constraints have a clear separation between variable on the left, and constants on the right. 

All constraints are added to the `.constraints` container from where all assigned constraints can be accessed.

In [ ]:
m.constraints

In [ ]:
m.constraints["my-constraint"]

## Coordinate Alignment in Constraints

As an alternative to the ``<=``, ``>=``, ``==`` operators, linopy provides ``.le()``, ``.ge()``, and ``.eq()`` methods on variables and expressions. These methods accept a ``join`` parameter (``"inner"``, ``"outer"``, ``"left"``, ``"right"``) for explicit control over how coordinates are aligned when creating constraints. See the :doc:`coordinate-alignment` guide for details.

## CSR Backend (Advanced)

By default, linopy stores each constraint as an `xarray.Dataset` (`Constraint`). This is flexible and allows full label-based indexing, but can use significant memory when constraints have many terms.

For large models, linopy provides an alternative **CSR backend** via the `CSRConstraint` class. It stores the constraint coefficients as a [scipy CSR sparse matrix](https://docs.scipy.org/doc/scipy/reference/generated/scipy.sparse.csr_array.html) with flat numpy arrays for the right-hand side and signs. This can reduce memory usage by up to **90%** and speeds up matrix generation for direct solver APIs by **30–120x**.

`CSRConstraint` is **immutable** — once frozen, the constraint data cannot be modified in place. You can always convert back to the mutable xarray-backed `Constraint` if needed.

### Freezing individual constraints

Pass `freeze=True` to `add_constraints` to store a single constraint in CSR format:

In [ ]:
import numpy as np

from linopy import Model

m2 = Model()
y = m2.add_variables(coords=[np.arange(100)], name="y")

m2.add_constraints(y <= 10, name="upper", freeze=True)

print(type(m2.constraints["upper"]))
m2.constraints["upper"]

### Freezing all constraints globally

Set `freeze_constraints=True` on the `Model` to automatically freeze every constraint added via `add_constraints`:

In [ ]:
m3 = Model(freeze_constraints=True)
z = m3.add_variables(coords=[np.arange(50)], name="z")
m3.add_constraints(z >= 0, name="lower")
m3.add_constraints(z <= 100, name="upper")

print(type(m3.constraints["lower"]))
print(type(m3.constraints["upper"]))

### Converting between representations

Use `.freeze()` and `.mutable()` to convert between the two representations. The conversion is lossless:

In [ ]:
frozen = m3.constraints["lower"]
print(f"Frozen type:  {type(frozen).__name__}")

thawed = frozen.mutable()
print(f"Mutable type: {type(thawed).__name__}")

refrozen = thawed.freeze()
print(f"Re-frozen type: {type(refrozen).__name__}")

### API differences from `Constraint`

`CSRConstraint` deliberately exposes a narrower API than the xarray-backed `Constraint`:

- **No in-place mutation.** Setters such as `con.coeffs = ...`, `con.vars = ...`, `con.sign = ...`, `con.rhs = ...`, and `con.lhs = ...` are only available on `Constraint`.
- **No label-based indexing.** `con.loc[...]` is only available on `Constraint`.
- **Accessing `.coeffs` / `.vars` triggers reconstruction.** On a `CSRConstraint` these properties rebuild the full xarray `Dataset` on demand and emit a `PerformanceWarning`. For solver-oriented workflows prefer `con.to_matrix()` or work with the CSR data directly.

If you need any of the above, call `.mutable()` first to get a `Constraint`:

```python
con = m.constraints["my_constraint"].mutable()
con.loc[{"time": 0}]  # label-based indexing now available
con.rhs = 5           # mutation now available
```

### When to use the CSR backend

The CSR backend is most beneficial when:

- Your model has **many constraints with many terms**.
- **Memory** is a bottleneck.
- You use a **direct solver API** (e.g. HiGHS, Gurobi Python bindings) rather than file-based I/O.

For small models the overhead is negligible and the default xarray-backed `Constraint` is perfectly fine.

Additionally, if you don't need variable and constraint names in the solver (e.g. for batch solves), you can disable name export for extra speed:

```python
m = Model(freeze_constraints=True, set_names_in_solver_io=False)
```